#### `What is my total spending on food this month?`

In [6]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

In [7]:
import aisuite as ai

client = ai.Client()

In [8]:
from utils.utils import print_html, get_schema 

print_html(get_schema('personal_finance.db'))

In [16]:
## schema of databse

schema = """
table name: transactions
id (INTEGER)
date (DATE)
category (VARCHAR(30))
type (VARCHAR(30))
amount (REAL)
description (VARCHAR(100))
payment_method (VARCHAR(30))
"""

question = "What is my total spending on food this month?"

prompt = f"""
You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

Schema: 
{schema}

user Question:
{question}

Respond with the SQL only.
"""


model = "openai:gpt-4.1"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
        )


sql_1 = response.choices[0].message.content.strip()


In [17]:
print_html(sql_1, title="SQL Query V1")

In [18]:
sql_1

"```sql\nSELECT SUM(amount) AS total_spending\nFROM transactions\nWHERE category = 'food'\n  AND type = 'expense'\n  AND strftime('%Y-%m', date) = strftime('%Y-%m', 'now');\n```"

In [20]:
import re

match = re.search(r"```sql\s*(.*?)\s*```", sql_1, re.DOTALL)

if match:
    sql_query = match.group(1)
    print(sql_query)

SELECT SUM(amount) AS total_spending
FROM transactions
WHERE category = 'food'
  AND type = 'expense'
  AND strftime('%Y-%m', date) = strftime('%Y-%m', 'now');


In [22]:
import sqlite3


conn = sqlite3.connect("personal_finance.db")
cur = conn.cursor()
cur.execute(sql_query)
rows = cur.fetchall()
conn.close()
# "table name: transactions\n" + "\n".join([f"{r[1]} ({r[2]})" for r in rows])
